# Train CodeBERT for AI Security Code Review

This is a **standalone** notebook. It will download the `DetectVul/devign` dataset directly from Hugging Face, fine-tune `microsoft/codebert-base`, and save the model weights and metrics.

### Instructions:
1. Go to **Runtime â†’ Change runtime type** and select **T4 GPU**.
2. Run all cells.
3. Download the resulting `codebert-vuln.zip` file.
4. Extract it into your project's `models/codebert-vuln/` directory.

In [ ]:
!pip install -q torch transformers datasets scikit-learn accelerate huggingface_hub matplotlib

In [ ]:
import torch
import numpy as np
import json
from datasets import Dataset, load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
import matplotlib.pyplot as plt
import shutil
import os

assert torch.cuda.is_available(), 'Please enable T4 GPU: Runtime â†’ Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Load and Prepare Dataset

In [ ]:
MODEL_NAME = "microsoft/codebert-base"
HF_DATASET = "DetectVul/devign"
MAX_LENGTH = 512

print(f"Loading dataset: {HF_DATASET}")
raw = load_dataset(HF_DATASET)
val_key = "validation" if "validation" in raw else "val"

def to_ds(split):
    return Dataset.from_dict({
        "text": split["func"],
        "labels": [int(x) for x in split["target"]]
    })

train_ds = to_ds(raw["train"])
val_ds = to_ds(raw[val_key])
test_ds = to_ds(raw["test"])

print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

## 2. Tokenize

*Change 2 & 3: Added class weighting cells and switched to dynamic padding via DataCollatorWithPadding to save computation.*

In [ ]:
from collections import Counter
import torch

label_counts = Counter(train_ds["labels"])
total = sum(label_counts.values())
class_weights = torch.tensor([
    total / (2 * label_counts[0]),
    total / (2 * label_counts[1])
], dtype=torch.float).cuda()

print(f"Class distribution: {label_counts}")
print(f"Class weights: {class_weights}")

In [ ]:
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_ds(ds):
    def _tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    out = ds.map(_tok, batched=True, remove_columns=["text"])
    out = out.rename_column("labels", "labels")
    out.set_format("torch")
    return out

print("Tokenizing datasets...")
train_tok = tokenize_ds(train_ds)
val_tok = tokenize_ds(val_ds)
test_tok = tokenize_ds(test_ds)

## 3. Train Model

*Change 1, 4 & 5: Lowered learning rate to 1e-5, added EarlyStoppingCallback, reduced batch size to 8 with gradient accumulation steps 2, and used WeightedTrainer.*

In [ ]:
from transformers import EarlyStoppingCallback

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": p,
        "recall": r,
        "f1": f1,
    }

output_dir = "codebert-vuln"

training_args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting training loop...")
trainer.train()

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

## 4. Evaluate & Visualize

*Change 6: Added per-class classification report to inspect class imbalance handling.*

In [ ]:
print("Evaluating on test split...")
pred = trainer.predict(test_tok)
preds = np.argmax(pred.predictions, axis=1)
labels = pred.label_ids

accuracy = float(accuracy_score(labels, preds))
p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
cm = confusion_matrix(labels, preds).tolist()

metrics = {
    "model": MODEL_NAME,
    "accuracy": accuracy,
    "precision": float(p),
    "recall": float(r),
    "f1": float(f1),
    "confusion_matrix": cm
}

os.makedirs("results", exist_ok=True)
with open("results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nMetrics: F1={f1:.4f} Acc={accuracy:.4f}")
print("\nPer-class report:")
print(classification_report(labels, preds, target_names=["Safe", "Vulnerable"]))

# Plot confusion matrix
cm_np = np.array(cm)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_np, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm_np[i, j], ha='center', va='center', color='white' if cm_np[i,j] > cm_np.max()/2 else 'black')
plt.show()

## 5. Export Weights

In [ ]:
from google.colab import files

shutil.make_archive('codebert-vuln', 'zip', 'codebert-vuln')
print("Downloading codebert-vuln.zip...")
files.download('codebert-vuln.zip')
files.download('results/metrics.json')